[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ELTE-DSED/Intro-Data-Security/blob/main/module_08_defenses/Lab_8b_Federated_Learning_and_Adversarial_Training.ipynb)

# **Lab 8b: Federated Learning and Adversarial Training**

**Course:** Introduction to Data Security  
**Module 8:** Defenses  
**Estimated Time:** 90 minutes

---

This lab shows how hospitals can collaborate on a shared medical-imaging task without pooling raw patient data, and how adversarial training can improve model robustness in this federated setting.

<div align="center">
  <img src="https://raw.githubusercontent.com/ELTE-DSED/Intro-Data-Security/main/module_08_defenses/images/federation_figure.jpg" width=60% alt="federation figure">
</div>


Medical data is sensitive, so hospitals should not centralize raw patient records if collaboration can be achieved in a privacy-preserving way. In this notebook, we use a public chest X-ray benchmark to model that setting and show how federated learning reduces data exposure.

### Setup

Before we start, please run the following to make sure that your environment is correctly setup.

In [ ]:
%pip install --quiet "flwr[simulation]" medmnist


### Dependencies

In [ ]:
import flwr as fl
from flwr.common import ndarrays_to_parameters
import medmnist
from medmnist import INFO
import ray
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from collections import OrderedDict, Counter, defaultdict
import random
import warnings
import logging
logging.getLogger("ray").setLevel(logging.WARNING)
warnings.filterwarnings('ignore', message='.*start_simulation.*deprecated.*')

np.random.seed(42)
torch.manual_seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

NUM_WORKERS = 0
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


### Preparing the input data

Loading PneumoniaMNIST (public chest X-ray dataset)


In [ ]:
DataClass = getattr(medmnist, INFO["pneumoniamnist"]["python_class"])
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

train_dataset = DataClass(split="train", transform=transform, download=True)
test_dataset  = DataClass(split="test",  transform=transform, download=True)

In [ ]:
def _extract_labels(source):
    """Return a flat int array of labels for MedMNIST dataset."""
    base = source.dataset if isinstance(source, Subset) else source
    indices = np.asarray(source.indices if isinstance(source, Subset) else np.arange(len(base)))
    return base.labels[indices].squeeze(axis=-1).astype(int)

In [ ]:
def _stratified_split_indices(indices, labels, split_fraction=0.1, seed=2026):
    indices = np.asarray(indices)
    labels = np.asarray(labels)

    if len(indices) <= 1:
        return indices, indices

    rng = np.random.default_rng(seed)
    train_parts = []
    eval_parts = []

    for label in np.unique(labels):
        class_indices = indices[labels == label].copy()
        rng.shuffle(class_indices)

        if len(class_indices) == 1:
            eval_count = 0
        else:
            eval_count = int(round(len(class_indices) * split_fraction))
            eval_count = max(1, min(eval_count, len(class_indices) - 1))

        eval_parts.append(class_indices[:eval_count])
        train_parts.append(class_indices[eval_count:])

    train_idx = np.concatenate(train_parts).astype(int, copy=False)
    eval_idx = np.concatenate(eval_parts).astype(int, copy=False)

    if len(eval_idx) == 0:
        moved_index = train_idx[0]
        eval_idx = np.array([moved_index], dtype=int)
        train_idx = train_idx[1:]
    elif len(train_idx) == 0:
        moved_index = eval_idx[0]
        train_idx = np.array([moved_index], dtype=int)
        eval_idx = eval_idx[1:]

    rng.shuffle(train_idx)
    rng.shuffle(eval_idx)
    return train_idx, eval_idx

In [ ]:

def split_for_server_eval(dataset, eval_fraction=0.1, seed=2026):
    n = len(dataset)
    if n <= 1:
        return dataset, dataset

    indices = np.asarray(dataset.indices if isinstance(dataset, Subset) else np.arange(n))
    labels = _extract_labels(dataset)
    train_idx, eval_idx = _stratified_split_indices(indices, labels, split_fraction=eval_fraction, seed=seed)

    if isinstance(dataset, Subset):
        return (Subset(dataset.dataset, train_idx.tolist()),
                Subset(dataset.dataset, eval_idx.tolist()))

    return Subset(dataset, train_idx.tolist()), Subset(dataset, eval_idx.tolist())

In [ ]:
train_data, server_eval_data = split_for_server_eval(train_dataset, eval_fraction=0.1, seed=2026)

server_eval_loader = DataLoader(server_eval_data, batch_size=128, shuffle=False, num_workers=NUM_WORKERS)
test_loader        = DataLoader(test_dataset,     batch_size=128, shuffle=False, num_workers=NUM_WORKERS)

In [ ]:
sample_x, _ = train_data[0]
input_shape  = tuple(sample_x.shape)

label_meta = INFO["pneumoniamnist"].get("label")
if isinstance(label_meta, dict):
    class_names = [str(label_meta[k]) for k in sorted(label_meta, key=lambda k: int(k) if str(k).isdigit() else k)]
elif isinstance(label_meta, list):
    class_names = [str(x) for x in label_meta]
else:
    class_names = []

train_labels = _extract_labels(train_data)
unique_classes = np.unique(train_labels)
num_classes    = unique_classes.size
class_desc     = " / ".join(class_names) if class_names else "unknown"

print(f"Train samples (client pool): {len(train_data)}")
print(f"Server eval samples: {len(server_eval_data)}, Test samples (final only): {len(test_dataset)}")
print(f"Input shape: {input_shape} - grayscale chest X-ray images")
print(f"Classes: {num_classes} ({class_desc})")


### Class-distribution bar chart

In [ ]:
counts = Counter(train_labels.tolist())
labels_sorted = sorted(counts)
label_names  = [class_names[i] if i < len(class_names) else str(i) for i in labels_sorted]

plt.figure(figsize=(6, 4))
plt.bar(label_names, [counts[i] for i in labels_sorted], color="#2ecc71")
plt.title("Client training pool class distribution")
plt.ylabel("Count")
plt.xlabel("Class")
plt.tight_layout()
plt.show()

### example image per class

In [ ]:


indices_by_class = defaultdict(list)
for i, lbl in enumerate(train_labels):
    indices_by_class[int(lbl)].append(i)

fig, axes = plt.subplots(1, num_classes, figsize=(3 * num_classes, 3))
axes = [axes] if num_classes == 1 else list(axes)

for ax, cls in zip(axes, range(num_classes)):
    pool = indices_by_class.get(cls, [])
    if not pool:
        ax.axis("off")
        continue
    img, lbl = train_data[random.choice(pool)]
    img_disp = img.squeeze().numpy() * 0.5 + 0.5   # un-normalize
    ax.imshow(img_disp, cmap="gray")
    title = (class_names[cls] if cls < len(class_names) else str(cls)) + f"\nLabel={int(np.array(lbl).squeeze())}"
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

### Model Definition

In [ ]:
class Net(nn.Module):
    def __init__(self, num_classes=2):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in loader:
            data = data.to(device)
            target = target.to(device).long().view(-1)
            output = model(data)
            pred = output.argmax(dim=1)
            correct += (pred == target).sum().item()
            total += target.size(0)
    return 100.0 * correct / max(total, 1)

In [ ]:
def fgsm_attack(model, data, target, epsilon=0.12):
    model.eval()
    x = data.detach().clone().requires_grad_(True)
    y = target.detach()
    logits = model(x)
    loss = nn.CrossEntropyLoss()(logits, y)
    grad = torch.autograd.grad(loss, x, only_inputs=True)[0]
    x_adv = x + epsilon * grad.sign()
    return torch.clamp(x_adv, -1.0, 1.0).detach()

In [ ]:
def evaluate_under_fgsm(model, loader, epsilon=0.12):
    model.eval()
    correct = 0
    total = 0
    for data, target in loader:
        data = data.to(device)
        target = target.to(device).long().view(-1)
        adv = fgsm_attack(model, data, target, epsilon=epsilon)
        with torch.no_grad():
            output = model(adv)
            pred = output.argmax(dim=1)
        correct += (pred == target).sum().item()
        total += target.size(0)
    return 100.0 * correct / max(total, 1)


## Federated Learning and Robustness Experiments <a id="fl"></a>

Let's start with the data. Federated learning requires a federated data set,
i.e., a collection of data from multiple users. Federated data is typically
non-[i.i.d.](https://en.wikipedia.org/wiki/Independent_and_identically_distributed_random_variables),
which poses a unique set of challenges.


In [ ]:
def partition(dataset, n_clients=5, non_iid=False):
    """Split the medical dataset into client partitions representing hospitals."""
    base_indices = np.asarray(dataset.indices if isinstance(dataset, Subset) else np.arange(len(dataset)))
    labels = _extract_labels(dataset)

    if non_iid:
        sorted_indices = base_indices[np.argsort(labels)]
        client_splits = np.array_split(sorted_indices, n_clients)
    else:
        rng = np.random.default_rng(42)
        client_buckets = [[] for _ in range(n_clients)]
        for label in np.unique(labels):
            class_indices = base_indices[labels == label].copy()
            rng.shuffle(class_indices)
            for position, sample_idx in enumerate(class_indices):
                client_buckets[position % n_clients].append(int(sample_idx))
        client_splits = [np.array(split, dtype=int) for split in client_buckets]

    base_dataset = dataset.dataset if isinstance(dataset, Subset) else dataset
    return [Subset(base_dataset, split.tolist()) for split in client_splits]

We will run two experiments:
- Baseline federated training (standard local SGD)
- Federated adversarial training (FGSM-based local robust optimization)

**Expected pattern:** baseline FL often reaches higher clean accuracy, while adversarial training usually improves FGSM robustness at some clean-accuracy cost.

In [ ]:
def get_parameters(model):
    return [val.detach().cpu().numpy() for _, val in model.state_dict().items()]


def set_parameters(model, parameters):
    params_dict = zip(model.state_dict().keys(), parameters)
    state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
    model.load_state_dict(state_dict, strict=True)


def weighted_average(metrics):
    if not metrics:
        return {}
    total_examples = sum(num_examples for num_examples, _ in metrics)
    return {
        'val_accuracy': float(
            sum(num_examples * float(m.get('val_accuracy', 0.0)) for num_examples, m in metrics) / max(total_examples, 1)
        )
    }


def clone_ndarrays(arrays):
    return [np.array(a, copy=True) for a in arrays]


def extract_metric_series(history_tuples, drop_round_zero=False):
    pairs = [(int(r), float(v)) for r, v in history_tuples]
    if drop_round_zero:
        pairs = [(r, v) for r, v in pairs if r > 0]
    rounds = [r for r, _ in pairs]
    values = [v for _, v in pairs]
    return rounds, values

In [ ]:
def run_flower_fedavg(
    client_train_loaders,
    client_val_loaders,
    rounds=5,
    local_epochs=1,
    lr=0.01,
    adv_training=False,
    adv_epsilon=0.12,
    adv_weight=0.5,
    eval_epsilon=0.12,
    initial_parameters=None,
    keep_initialised=False,
):
    if initial_parameters is None:
        seed_model = Net(num_classes=num_classes).to(device)
        initial_parameters = get_parameters(seed_model)
    initial_parameters = clone_ndarrays(initial_parameters)

    global_model = Net(num_classes=num_classes).to(device)
    set_parameters(global_model, initial_parameters)
    latest_parameters = clone_ndarrays(initial_parameters)

    initial_parameters_fl = ndarrays_to_parameters(initial_parameters)

    def client_fn(context):
        partition_id = int(context.node_config['partition-id'])
        model = Net(num_classes=num_classes).to(device)
        client = FlowerHospitalClient(model, client_train_loaders[partition_id], client_val_loaders[partition_id])
        return client.to_client()

    def on_fit_config_fn(server_round):
        return {
            'local_epochs': local_epochs,
            'lr': lr,
            'adv_training': adv_training,
            'adv_epsilon': adv_epsilon,
            'adv_weight': adv_weight,
        }

    def evaluate_fn(server_round, parameters, config):
        nonlocal latest_parameters
        latest_parameters = clone_ndarrays(parameters)
        set_parameters(global_model, parameters)

        # Centralized server-side validation each round (not test set).
        clean_acc = evaluate(global_model, server_eval_loader)
        fgsm_acc = evaluate_under_fgsm(global_model, server_eval_loader, epsilon=eval_epsilon)
        loss = 1.0 - clean_acc / 100.0
        return float(loss), {'clean_accuracy': float(clean_acc), 'fgsm_accuracy': float(fgsm_acc)}

    strategy = fl.server.strategy.FedAvg(
        fraction_fit=1.0,
        fraction_evaluate=1.0,
        min_fit_clients=len(client_train_loaders),
        min_evaluate_clients=len(client_train_loaders),
        min_available_clients=len(client_train_loaders),
        on_fit_config_fn=on_fit_config_fn,
        evaluate_fn=evaluate_fn,
        fit_metrics_aggregation_fn=weighted_average,
        evaluate_metrics_aggregation_fn=weighted_average,
        initial_parameters=initial_parameters_fl,
    )

    history = fl.simulation.start_simulation(
        client_fn=client_fn,
        num_clients=len(client_train_loaders),
        config=fl.server.ServerConfig(num_rounds=rounds),
        strategy=strategy,
        client_resources={'num_cpus': 1},
        ray_init_args={'include_dashboard': False, 'ignore_reinit_error': True},
    )

    metric_rounds, clean_history = extract_metric_series(
        history.metrics_centralized.get('clean_accuracy', []), drop_round_zero=True
    )
    fgsm_rounds, fgsm_history = extract_metric_series(
        history.metrics_centralized.get('fgsm_accuracy', []), drop_round_zero=True
    )
    val_rounds, val_history = extract_metric_series(
        history.metrics_distributed_fit.get('val_accuracy', []), drop_round_zero=False
    )

    set_parameters(global_model, latest_parameters)
    final_test_clean = evaluate(global_model, test_loader)
    final_test_fgsm = evaluate_under_fgsm(global_model, test_loader, epsilon=eval_epsilon)

    if not keep_initialised:
        ray.shutdown()

    return {
        'backend': 'flower_simulation',
        'rounds': metric_rounds,
        'server_eval_clean_accuracy': clean_history,
        'server_eval_fgsm_accuracy': fgsm_history,
        'client_val_accuracy': val_history,
        'client_val_rounds': val_rounds,
        'final_test_clean_accuracy': float(final_test_clean),
        'final_test_fgsm_accuracy': float(final_test_fgsm),
    }


In [ ]:
def local_train(model, loader, epochs=1, lr=0.01, adv_training=False, adv_epsilon=0.12, adv_weight=0.5):
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    for _ in range(epochs):
        for data, target in loader:
            data = data.to(device)
            target = target.to(device).long().view(-1)

            optimizer.zero_grad()
            logits = model(data)
            loss_clean = criterion(logits, target)

            if adv_training:
                adv = fgsm_attack(model, data, target, epsilon=adv_epsilon)
                logits_adv = model(adv)
                loss_adv = criterion(logits_adv, target)
                loss = (1.0 - adv_weight) * loss_clean + adv_weight * loss_adv
            else:
                loss = loss_clean

            loss.backward()
            optimizer.step()

In [ ]:
class FlowerHospitalClient(fl.client.NumPyClient):
    def __init__(self, model, train_loader, val_loader):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader

    def get_parameters(self, config):
        return get_parameters(self.model)

    def fit(self, parameters, config):
        set_parameters(self.model, parameters)
        local_epochs = int(config.get('local_epochs', 1))
        lr = float(config.get('lr', 0.01))
        adv_training = bool(config.get('adv_training', False))
        adv_epsilon = float(config.get('adv_epsilon', 0.12))
        adv_weight = float(config.get('adv_weight', 0.5))

        local_train(
            self.model,
            self.train_loader,
            epochs=local_epochs,
            lr=lr,
            adv_training=adv_training,
            adv_epsilon=adv_epsilon,
            adv_weight=adv_weight,
        )

        val_acc = evaluate(self.model, self.val_loader)
        return get_parameters(self.model), len(self.train_loader.dataset), {'val_accuracy': float(val_acc)}

    def evaluate(self, parameters, config):
        set_parameters(self.model, parameters)
        val_acc = evaluate(self.model, self.val_loader)
        loss = 1.0 - val_acc / 100.0
        return float(loss), len(self.val_loader.dataset), {'val_accuracy': float(val_acc)}


In [ ]:
def make_client_loaders(client_subsets, batch_size=64, val_fraction=0.2, seed=42):
    """train/validation loaders for each client subset."""
    train_loaders = []
    val_loaders = []

    for client_idx, subset in enumerate(client_subsets):
        local_indices = np.asarray(subset.indices)
        local_labels = _extract_labels(subset)
        train_indices, val_indices = _stratified_split_indices(
            local_indices,
            local_labels,
            split_fraction=val_fraction,
            seed=seed + client_idx,
        )

        train_subset = Subset(subset.dataset, train_indices.tolist())
        val_subset = Subset(subset.dataset, val_indices.tolist())

        train_loaders.append(DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS))
        val_loaders.append(DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS))

    return train_loaders, val_loaders

In [ ]:
# Partition hospitals
NUM_CLIENTS = 5
clients = partition(train_data, n_clients=NUM_CLIENTS, non_iid=False)
client_train_loaders, client_val_loaders = make_client_loaders(clients, batch_size=64, val_fraction=0.2, seed=42)
print(f"Created {len(clients)} hospital clients.")

# Global training config
ROUNDS = 8
LOCAL_EPOCHS = 2
LR = 0.01
FGSM_EVAL_EPSILON = 0.2

# Use identical initialization for fair baseline-vs-robust comparison
reference_model = Net(num_classes=num_classes).to(device)
shared_initial_parameters = get_parameters(reference_model)

### Baseline Federated Training

In [ ]:
baseline_results = run_flower_fedavg(
    client_train_loaders,
    client_val_loaders,
    rounds=ROUNDS,
    local_epochs=LOCAL_EPOCHS,
    lr=LR,
    adv_training=False,
    eval_epsilon=FGSM_EVAL_EPSILON,
    initial_parameters=shared_initial_parameters,
    keep_initialised=True,
)

### Federated Adversarial Training

In [ ]:
robust_results = run_flower_fedavg(
    client_train_loaders,
    client_val_loaders,
    rounds=ROUNDS,
    local_epochs=LOCAL_EPOCHS,
    lr=LR,
    adv_training=True,
    adv_epsilon=0.12,
    adv_weight=0.5,
    eval_epsilon=FGSM_EVAL_EPSILON,
    initial_parameters=shared_initial_parameters,
)

In [ ]:
rounds_axis = baseline_results['rounds']

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(rounds_axis, baseline_results['server_eval_clean_accuracy'], marker='o', linewidth=2.2, label='Baseline FL (server eval, clean)')
axes[0].plot(rounds_axis, robust_results['server_eval_clean_accuracy'], marker='s', linewidth=2.2, label='AdvTr FL (server eval, clean)')
axes[0].set_xlabel('Federated Round')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Centralized Server Eval: Clean')
axes[0].grid(alpha=0.3, linestyle='--')
axes[0].set_ylim(0, 100)
axes[0].legend(fontsize=9)

axes[1].plot(rounds_axis, baseline_results['server_eval_fgsm_accuracy'], marker='o', linewidth=2.2, label='Baseline FL (server eval, FGSM)')
axes[1].plot(rounds_axis, robust_results['server_eval_fgsm_accuracy'], marker='s', linewidth=2.2, label='AdvTr FL (server eval, FGSM)')
axes[1].set_xlabel('Federated Round')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Centralized Server Eval: FGSM')
axes[1].grid(alpha=0.3, linestyle='--')
axes[1].set_ylim(0, 100)
axes[1].legend(fontsize=9)

plt.suptitle('Federated Learning: Standard vs Adversarial Training', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

final_clean_gain = robust_results['final_test_clean_accuracy'] - baseline_results['final_test_clean_accuracy']
final_fgsm_gain = robust_results['final_test_fgsm_accuracy'] - baseline_results['final_test_fgsm_accuracy']
print(f"Final TEST clean accuracy gain (robust - baseline): {final_clean_gain:+.2f}%")
print(f"Final TEST FGSM accuracy gain (robust - baseline): {final_fgsm_gain:+.2f}%")


## Exercises

1. Set `FGSM_EVAL_EPSILON` to `0.05`, `0.1`, `0.2`, and `0.3`, then compare baseline vs adversarial training final test accuracy.
2. Run adversarial training with `adv_weight` in `{0.3, 0.5, 0.7}`, and explain the clean-robustness tradeoff.
3. Change hospital partitioning to `non_iid=True`, rerun both methods, and compare convergence behavior against `non_iid=False`.
4. Run with `NUM_CLIENTS` in `{3, 5, 8}` and discuss how client count affects round-to-round stability and robustness.
5. Compare `(ROUNDS=8, LOCAL_EPOCHS=2)` against `(ROUNDS=4, LOCAL_EPOCHS=4)` and analyze which setting gives better final robustness under similar local work.
6. Change `eval_fraction` in `split_for_server_eval` from `0.1` to `0.2` and discuss whether conclusions about baseline vs adversarial training stay consistent.
